## PARTIE 2 : Deep Learning - Classification d'images

In [14]:
import keras
import numpy as np
import pandas as pd
from keras.src.backend.tensorflow.nn import softmax
from sklearn.preprocessing import StandardScaler
from tensorflow.python.ops.nn_impl import relu_layer

fichier = "WangSignatures.xlsx"

noms_onglets = {
    "JCD": "WangSignaturesJCD",
    "PHOG": "WangSignaturesPHOG",
    "CEDD": "WangSignaturesCEDD",
    "FCTH": "WangSignaturesFCTH",
    "FuzzyColor": "WangSignaturesFuzzyColorHistogr"
}

# Chargement des cinq descripteurs
descripteurs = {}

for nom, onglet in noms_onglets.items():
    df = pd.read_excel(
        fichier,
        sheet_name=onglet,
        header=None,
        index_col=0
    )

    descripteurs[nom] = df

    print(f"{nom} : {df.shape}")

# Vérification des noms d'images
ordre_images = descripteurs["JCD"].index

assert len(ordre_images) == 1000
assert ordre_images.is_unique

for nom, df in descripteurs.items():
    assert df.index.is_unique
    assert len(df) == 1000
    assert set(df.index) == set(ordre_images)

    descripteurs[nom] = df.loc[ordre_images]

# Construction des labels
labels = np.array([
    int(nom[:-4]) // 100
    for nom in ordre_images
])

print("\nRépartition des classes :")
classes, effectifs = np.unique(labels, return_counts=True)

for classe, effectif in zip(classes, effectifs):
    print(f"Classe {classe} : {effectif} images")

# Conversion en matrices NumPy
X_descripteurs = {}

for nom, df in descripteurs.items():
    X_descripteurs[nom] = df.to_numpy(dtype=float)

# Concaténation des cinq descripteurs
X_complet = np.concatenate(
    list(X_descripteurs.values()),
    axis=1
)

# Vérifications finales
assert X_complet.shape[0] == len(labels)
assert np.isfinite(X_complet).all()

print("\nDimensions des matrices :")

for nom, X in X_descripteurs.items():
    print(f"{nom} : {X.shape}")

print(f"\nTous les descripteurs : {X_complet.shape}")
print(f"Labels : {labels.shape}")

JCD : (1000, 168)
PHOG : (1000, 255)
CEDD : (1000, 144)
FCTH : (1000, 192)
FuzzyColor : (1000, 125)

Répartition des classes :
Classe 0 : 100 images
Classe 1 : 100 images
Classe 2 : 100 images
Classe 3 : 100 images
Classe 4 : 100 images
Classe 5 : 100 images
Classe 6 : 100 images
Classe 7 : 100 images
Classe 8 : 100 images
Classe 9 : 100 images

Dimensions des matrices :
JCD : (1000, 168)
PHOG : (1000, 255)
CEDD : (1000, 144)
FCTH : (1000, 192)
FuzzyColor : (1000, 125)

Tous les descripteurs : (1000, 884)
Labels : (1000,)


## Partie 2.1.2

In [ ]:
from sklearn.model_selection import train_test_split

# ÉTAPE 2 : Séparation stratifiée en train/test (80% / 20%)
indices = np.arange(len(labels))

idx_train_val, idx_test = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels, # Conserve la proportion de chaque classe
    random_state=42
)

# Séparation supplémentaire pour avoir un set de validation (utilisé pour l'EarlyStopping)
idx_train, idx_val = train_test_split(
    idx_train_val,
    test_size=0.2,
    stratify=labels[idx_train_val],
    random_state=42
)

y_train = labels[idx_train]
y_val = labels[idx_val]
y_test = labels[idx_test]

print(f"Images d'entraînement : {len(idx_train)}")
print(f"Images de validation : {len(idx_val)}")
print(f"Images de test : {len(idx_test)}")

In [ ]:
def creer_modele(nb_entrees):
    model = keras.Sequential()
    model.add(keras.Input(shape=(nb_entrees,)))
    model.add(keras.Dense(64, activation="relu"))
    model.add(keras.Dense(32, activation="relu"))
    model.add(keras.Dense(10, activation="softmax"))

    model.compile(
        optimize="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
# 1. Création d'un dictionnaire unique regroupant les 6 configurations
ensembles = {**X_descripteurs, "Complet": X_complet}

# Dictionnaire pour stocker les métriques et historiques de chaque modèle
resultats = {}

# 2. Boucle principale d'évaluation
for nom, X in ensembles.items():
    print(f"\n=== Entraînement sur le descripteur : {nom} ===")

    # 3. Extraction des sous-ensembles via les indices stratifiés (générés à l'Étape 2)
    X_train = X[idx_train]
    X_val = X[idx_val]

    scaler = StandardScaler()
    X_train.fit_transform()
    X_val.transform()

    model = creer_modele(X_train.shape[1])
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
    history = model.fit(
        X_train,
        y_train,
        batch_size=32,
        epochs=100,
        callbacks=[early_stopping],
        validation_data=(X_val, y_val),
        verbose=0
    )
    probas = model.predict(X_val)
    np.argmax(probas, axis=1)

    acc = accuracy_score(y_val, predictions)
    resultats[nom].append([acc, history, model])




